- Họ và Tên: Nguyễn Vạn Phúc Huy 
- MSSV: 23110163

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [ ]:
def add_intercept(x):
    """
      Dùng cho tất cả các bài trong file

    Args:
        x: 2D NumPy array.

    Returns:
        New matrix same as x with 1's in the 0th column.
    """
    new_x = np.zeros((x.shape[0], x.shape[1] + 1), dtype=x.dtype)
    new_x[:, 0] = 1
    new_x[:, 1:] = x

    return new_x


def load_dataset(csv_path, label_col='y', bias=False):
    """
    Dùng cho dataset ds1_train
    """


    # Validate label_col argument
    allowed_label_cols = ('y', 't')
    if label_col not in allowed_label_cols:
        raise ValueError('Invalid label_col: {} (expected {})'
                         .format(label_col, allowed_label_cols))

    # Load headers
    with open(csv_path, 'r') as csv_fh:
        headers = csv_fh.readline().strip().split(',')

    # Load features and labels
    x_cols = [i for i in range(len(headers)) if headers[i].startswith('x')]
    l_cols = [i for i in range(len(headers)) if headers[i] == label_col]
    inputs = np.loadtxt(csv_path, delimiter=',', skiprows=1, usecols=x_cols)
    labels = np.loadtxt(csv_path, delimiter=',', skiprows=1, usecols=l_cols)

    if inputs.ndim == 1:
        inputs = np.expand_dims(inputs, -1)

    if bias:
        inputs = add_intercept(inputs)

    return inputs, labels

In [ ]:
def ReLU(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

# Bài 1

In [ ]:
ds1_training_set_path = 'ds1_train.csv'
ds1_valid_set_path = 'ds1_valid.csv'

In [ ]:
NUM_EPOCHS = 100
LEARNING_RATE = 0.1

# Ta không thêm bias vào vì bản thân trong mạng đã có bias
X_train_raw, Y_train_raw = load_dataset(ds1_training_set_path, bias=False)
X_valid_raw, Y_valid_raw = load_dataset(ds1_valid_set_path, bias=False)

# Chuyển X từ (n_samples, n_features) → (n_features, n_samples)
# vì trong các công thức vectorized của neural network,
# mỗi cột là 1 mẫu (x^i). Nếu không .T thì mỗi hàng lại trở thành 1 mẫu,
# dẫn đến sai toàn bộ tích ma trận W @ X.

X_train = X_train_raw.T
X_valid = X_valid_raw.T

# Chuyển Y từ vector (n_samples,) → ma trận (1, n_samples)
# vì ta muốn Y có shape tương ứng với output layer shape (1, m_samples),
# phục vụ cho phép broadcast trong loss và backprop.

Y_train = Y_train_raw.reshape(1, -1)
Y_valid = Y_valid_raw.reshape(1, -1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2747 entries, 0 to 2746
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ORDERNUMBER           2747 non-null   int64  
 1   QUANTITYORDERED       2747 non-null   int64  
 2   PRICEEACH             2747 non-null   float64
 3   ORDERLINENUMBER       2747 non-null   int64  
 4   SALES                 2747 non-null   float64
 5   ORDERDATE             2747 non-null   object 
 6   DAYS_SINCE_LASTORDER  2747 non-null   int64  
 7   STATUS                2747 non-null   object 
 8   PRODUCTLINE           2747 non-null   object 
 9   MSRP                  2747 non-null   int64  
 10  PRODUCTCODE           2747 non-null   object 
 11  CUSTOMERNAME          2747 non-null   object 
 12  PHONE                 2747 non-null   object 
 13  ADDRESSLINE1          2747 non-null   object 
 14  CITY                  2747 non-null   object 
 15  POSTALCODE           

# Bài 2